# 新ベースライン（0.76574）からの 7 パターン実験

**参照**:
- スコア一覧・ベースライン: `docs/12_PUBLIC_SCORES_AND_NEW_BASELINE.md`
- 実験方針・各パターンの狙い: `docs/13_7PATTERNS_RATIONALE.md`
- 現状の実験まとめ（入出力・スキップ仕様）: `docs/14_CURRENT_EXPERIMENTS.md`

**ベースライン**: `submission_blend_bpr64_bpr128.csv` → Public **0.76574**（BPR64 と BPR128 の均等ブレンド）。

このノートでは同じ土台の上で **7 パターン** を試し、提出 CSV を保存する。

| # | 実験内容 | 提出ファイル |
|---|----------|-------------|
| 1 | ベースライン再現: BPR64 + BPR128 均等ブレンド | submission_blend_bpr64_bpr128.csv |
| 2 | ブレンド重み 0.4/0.6（BPR64 軽め） | submission_blend_bpr64_bpr128_046.csv |
| 3 | ブレンド重み 0.6/0.4（BPR128 軽め） | submission_blend_bpr64_bpr128_064.csv |
| 4 | 3 本ブレンド: BPR64 + BPR128 + ALS64 均等 | submission_blend_bpr64_bpr128_als64.csv |
| 5 | BPR64 シード平均（seed 42, 43, 44） | submission_blend_bpr64_seed_avg.csv |
| 6 | 2-hop count1 載せ: bpr64_count1 + BPR128 均等ブレンド | submission_blend_bpr64_count1_bpr128.csv |
| 7 | 3 本ブレンド重み付き（BPR64 0.35, BPR128 0.45, ALS64 0.2） | submission_blend_3way_weighted.csv |

## 1. セットアップ

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from lib.improvement_candidates import get_setup, run_atmacup_implicit
from lib.two_hop import run_experiment, TWO_HOP_REVIEW_COUNT
from lib.submission import save_submission, verify_submission

ctx = get_setup(seed=42, output_dir="outputs", use_best_pipeline=True)
print(f"Train: {len(ctx.train):,}, Test: {len(ctx.test):,}, Features: {len(ctx.features)}")
print(f"提出先: {ctx.submissions_dir}")

Train: 653,507, Test: 40,716, Features: 55
提出先: outputs/submissions


## 2. ブレンド用ヘルパー

In [2]:
def blend_two(path_a: Path, path_b: Path, out_name: str, w_a: float = 0.5, w_b: float = 0.5) -> dict:
    """2 本の提出 CSV を重み w_a, w_b で加重平均。"""
    if not path_a.exists() or not path_b.exists():
        return {"ok": False, "message": f"ファイル不足: {path_a.name} / {path_b.name}"}
    a = pd.read_csv(path_a)[["ID", "target"]].rename(columns={"target": "a"})
    b = pd.read_csv(path_b)[["ID", "target"]].rename(columns={"target": "b"})
    m = a.merge(b, on="ID")
    m["target"] = w_a * m["a"] + w_b * m["b"]
    # 行順を ctx.test に揃える（提出形式の整合性のため）
    m = m.set_index("ID").reindex(ctx.test["ID"]).reset_index()
    m["target"] = m["target"].fillna(0.5)
    out = ctx.submissions_dir / out_name
    save_submission(ctx.test, m["target"].values, out, sanitize=True)
    return verify_submission(out, ctx.test)


def blend_three(path_a: Path, path_b: Path, path_c: Path, out_name: str) -> dict:
    """3 本を均等 (1/3, 1/3, 1/3) でブレンド。"""
    return blend_three_weighted(path_a, path_b, path_c, out_name, 1/3, 1/3, 1/3)


def blend_three_weighted(
    path_a: Path, path_b: Path, path_c: Path, out_name: str,
    w_a: float, w_b: float, w_c: float,
) -> dict:
    """3 本を指定重みでブレンド。"""
    for p in (path_a, path_b, path_c):
        if not p.exists():
            return {"ok": False, "message": f"ファイル不足: {p.name}"}
    a = pd.read_csv(path_a)[["ID", "target"]].rename(columns={"target": "a"})
    b = pd.read_csv(path_b)[["ID", "target"]].rename(columns={"target": "b"})
    c = pd.read_csv(path_c)[["ID", "target"]].rename(columns={"target": "c"})
    m = a.merge(b, on="ID").merge(c, on="ID")
    m["target"] = w_a * m["a"] + w_b * m["b"] + w_c * m["c"]
    m = m.set_index("ID").reindex(ctx.test["ID"]).reset_index()
    m["target"] = m["target"].fillna(0.5)
    out = ctx.submissions_dir / out_name
    save_submission(ctx.test, m["target"].values, out, sanitize=True)
    return verify_submission(out, ctx.test)

## 3. 単体モデル作成（BPR64, BPR128, ALS64, bpr64_count1）

In [3]:
def _report(name: str, r: dict) -> None:
    if r.get("path"):
        p = r["path"]
        name_part = p.name if hasattr(p, "name") else Path(p).name
        print(f"  [{name}] → {name_part}  ({'OK' if r.get('ok') else r.get('message', '')})")
    else:
        print(f"  [{name}] スキップ: {r.get('message', '')}")

# BPR64 単体（ブレンド用）
path_bpr64 = ctx.submissions_dir / "submission_2hop_bpr64_only.csv"
if not path_bpr64.exists():
    r_bpr64 = run_experiment(ctx, "bpr64_only", bpr_factors=64)
    _report("BPR64 単体", r_bpr64)
else:
    print(f"  [BPR64 単体] 既存: {path_bpr64.name}")

# BPR128 単体（ブレンド用）
path_bpr128 = ctx.submissions_dir / "submission_2hop_bpr128_only.csv"
if not path_bpr128.exists():
    r_bpr128 = run_experiment(ctx, "bpr128_only", bpr_factors=128)
    _report("BPR128 単体", r_bpr128)
else:
    print(f"  [BPR128 単体] 既存: {path_bpr128.name}")

# ALS64 単体（3 本ブレンド用）
path_als64 = ctx.submissions_dir / "submission_atmacup_implicit_als64.csv"
if not path_als64.exists():
    r_als = run_atmacup_implicit(ctx, "als", 64, "als64")
    _report("ALS64 単体", r_als)
else:
    print(f"  [ALS64 単体] 既存: {path_als64.name}")

# BPR64 + 2-hop count1（パターン 6 用）
path_count1 = ctx.submissions_dir / "submission_2hop_bpr64_count1.csv"
if not path_count1.exists():
    r_count = run_experiment(ctx, "bpr64_count1", bpr_factors=64, use_2hop_cols=[TWO_HOP_REVIEW_COUNT])
    _report("BPR64+count1", r_count)
else:
    print(f"  [BPR64+count1] 既存: {path_count1.name}")

  [BPR64 単体] 既存: submission_2hop_bpr64_only.csv
  [BPR128 単体] 既存: submission_2hop_bpr128_only.csv
  [ALS64 単体] 既存: submission_atmacup_implicit_als64.csv
  [BPR64+count1] 既存: submission_2hop_bpr64_count1.csv


## 4. パターン 1〜4: ブレンド（ベースライン・重み付き・3 本）

In [4]:
# 1. ベースライン再現: BPR64 + BPR128 均等
r1 = blend_two(path_bpr64, path_bpr128, "submission_blend_bpr64_bpr128.csv", 0.5, 0.5)
_report("1 ベースライン再現", r1)

# 2. 重み 0.4 / 0.6（BPR64 軽め）
r2 = blend_two(path_bpr64, path_bpr128, "submission_blend_bpr64_bpr128_046.csv", 0.4, 0.6)
_report("2 ブレンド 0.4/0.6", r2)

# 3. 重み 0.6 / 0.4（BPR128 軽め）
r3 = blend_two(path_bpr64, path_bpr128, "submission_blend_bpr64_bpr128_064.csv", 0.6, 0.4)
_report("3 ブレンド 0.6/0.4", r3)

# 4. 3 本ブレンド: BPR64 + BPR128 + ALS64 均等
r4 = blend_three(path_bpr64, path_bpr128, path_als64, "submission_blend_bpr64_bpr128_als64.csv")
_report("4 3本ブレンド均等", r4)

  [1 ベースライン再現] → submission_blend_bpr64_bpr128.csv  (OK)
  [2 ブレンド 0.4/0.6] → submission_blend_bpr64_bpr128_046.csv  (OK)
  [3 ブレンド 0.6/0.4] → submission_blend_bpr64_bpr128_064.csv  (OK)
  [4 3本ブレンド均等] → submission_blend_bpr64_bpr128_als64.csv  (OK)


## 5. パターン 5: BPR64 シード平均（seed 42, 43, 44）

In [5]:
# seed 42 は既に submission_2hop_bpr64_only.csv として保存済み。43, 44 用に別名で保存。
path_bpr64_s43 = ctx.submissions_dir / "submission_2hop_bpr64_only_seed43.csv"
path_bpr64_s44 = ctx.submissions_dir / "submission_2hop_bpr64_only_seed44.csv"

for seed, path in [(43, path_bpr64_s43), (44, path_bpr64_s44)]:
    if path.exists():
        print(f"  [BPR64 seed {seed}] 既存: {path.name}")
        continue
    ctx_s = get_setup(seed=seed, output_dir="outputs", use_best_pipeline=True)
    r = run_experiment(ctx_s, f"bpr64_only_seed{seed}", bpr_factors=64)
    _report(f"BPR64 seed {seed}", r)

# 3 本を均等ブレンド（bpr64_only = seed42, _seed43, _seed44）
if path_bpr64_s43.exists() and path_bpr64_s44.exists():
    r5 = blend_three(path_bpr64, path_bpr64_s43, path_bpr64_s44, "submission_blend_bpr64_seed_avg.csv")
    _report("5 BPR64 シード平均", r5)
else:
    print("  [5 シード平均] seed43/44 のファイルが不足しています")

  0%|          | 0/100 [00:00<?, ?it/s]

  [BPR64 seed 43] → submission_2hop_bpr64_only_seed43.csv  (OK)


  0%|          | 0/100 [00:00<?, ?it/s]

  [BPR64 seed 44] → submission_2hop_bpr64_only_seed44.csv  (OK)
  [5 BPR64 シード平均] → submission_blend_bpr64_seed_avg.csv  (OK)


## 6. パターン 6: bpr64_count1 + BPR128 均等ブレンド

In [6]:
r6 = blend_two(path_count1, path_bpr128, "submission_blend_bpr64_count1_bpr128.csv", 0.5, 0.5)
_report("6 bpr64_count1 + BPR128", r6)

  [6 bpr64_count1 + BPR128] → submission_blend_bpr64_count1_bpr128.csv  (OK)


## 7. パターン 7: 3 本ブレンド重み付き（BPR64 0.35, BPR128 0.45, ALS64 0.2）

In [7]:
r7 = blend_three_weighted(
    path_bpr64, path_bpr128, path_als64,
    "submission_blend_3way_weighted.csv",
    0.35, 0.45, 0.2,
)
_report("7 3本ブレンド重み付き", r7)

  [7 3本ブレンド重み付き] → submission_blend_3way_weighted.csv  (OK)


## 8. 提出ファイル一覧

In [8]:
expected = [
    "submission_blend_bpr64_bpr128.csv",
    "submission_blend_bpr64_bpr128_046.csv",
    "submission_blend_bpr64_bpr128_064.csv",
    "submission_blend_bpr64_bpr128_als64.csv",
    "submission_blend_bpr64_seed_avg.csv",
    "submission_blend_bpr64_count1_bpr128.csv",
    "submission_blend_3way_weighted.csv",
]
for name in expected:
    p = ctx.submissions_dir / name
    print(f"  {'✓' if p.exists() else '✗'} {name}")

  ✓ submission_blend_bpr64_bpr128.csv
  ✓ submission_blend_bpr64_bpr128_046.csv
  ✓ submission_blend_bpr64_bpr128_064.csv
  ✓ submission_blend_bpr64_bpr128_als64.csv
  ✓ submission_blend_bpr64_seed_avg.csv
  ✓ submission_blend_bpr64_count1_bpr128.csv
  ✓ submission_blend_3way_weighted.csv
